In [ ]:
import datasets, fsspec
print("datasets version:", datasets.__version__)
print("fsspec version:", fsspec.__version__)

datasets version: 3.6.0
fsspec version: 2025.3.0


In [ ]:
pip install -U datasets

In [ ]:
import pandas as pd
import tensorflow as tf
from datasets import Dataset, load_dataset, DatasetDict, ClassLabel
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

# === A. Bersihkan CSV (hilangkan baris yang kosong atau NaN) ===
csv_input_path = "/content/data_text_final_status_normalized.csv"
df = pd.read_csv(csv_input_path)

# Hapus baris di mana 'text_final' atau 'Status_Normalized' kosong atau NaN
df = df.dropna(subset=["text_final", "Status_Normalized"])
df = df[df["text_final"].str.strip() != ""]
df = df[df["Status_Normalized"].str.strip() != ""]

# Simpan CSV bersih ke file baru
clean_csv_path = "/content/data_clean.csv"
df.to_csv(clean_csv_path, index=False)

In [ ]:
df

,text_final,Status_Normalized
0,hasil periksa fakta rahmah rsupn dr cipto mang...,HOAX
1,facebook indonesia boxbox indonesia nyaman sos...,BELUM DIVERIFIKASI
2,klinik misinformasi webinar rabu agustus wib s...,BELUM DIVERIFIKASI
3,indorelawan agustus simak view post instagram ...,BELUM DIVERIFIKASI
4,surakarta agustus selengkap view post instagra...,BELUM DIVERIFIKASI
...,...,...
16429,mafindo men debunk kompas com kompas com memua...,FAKTA
16430,update artikel menelusuri isu berkembang terka...,BELUM DIVERIFIKASI
16431,informasi terbaru artikel periksa fakta terbit...,BELUM DIVERIFIKASI
16432,direktur jenderal pencemaran kerusakan lingkun...,FAKTA


In [ ]:
# === B. Muat Dataset dari CSV dan Lakukan Label Encoding ===
# Muat CSV bersih sebagai Hugging Face Dataset
raw_ds = load_dataset("csv", data_files={"train": clean_csv_path})

# maka gunakan ClassLabel agar otomatis memapping string ke integer 0,1,2.
label_feature = ClassLabel(names=["HOAX", "BELUM DIVERIFIKASI", "FAKTA"])
raw_ds = raw_ds.cast_column("Status_Normalized", label_feature)

# Bagi menjadi train dan validation (90:10)
split = raw_ds["train"].train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"]
})

Generating train split: 0 examples [00:00, ? examples/s]

Casting the dataset:   0%|          | 0/16434 [00:00<?, ? examples/s]

In [ ]:
# === C. Tokenisasi ===
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_fn(examples):
    return tokenizer(
        examples["text_final"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Terapkan tokenisasi pada kedua split
tokenized_ds = dataset.map(
    preprocess_fn,
    batched=True,
    remove_columns=["text_final"]
)

# Ubah nama kolom label menjadi "labels"
tokenized_ds = tokenized_ds.rename_column("Status_Normalized", "labels")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/14790 [00:00<?, ? examples/s]

Map:   0%|          | 0/1644 [00:00<?, ? examples/s]

In [ ]:
# === D. Set Format ke TensorFlow ===
# Pastikan kolom: input_ids, attention_mask, labels (integer)
tokenized_ds.set_format(type="tensorflow", columns=["input_ids", "attention_mask", "labels"])

# === E. Konversi ke tf.data.Dataset dengan Casting Label ke tf.int32 ===
def to_tf_dataset(split_name):
    features = {
        "input_ids": tokenized_ds[split_name]["input_ids"],
        "attention_mask": tokenized_ds[split_name]["attention_mask"]
    }
    labels = tf.cast(tokenized_ds[split_name]["labels"], tf.int32)
    return tf.data.Dataset.from_tensor_slices((features, labels))

batch_size = 16
train_ds = (
    to_tf_dataset("train")
    .shuffle(1000, seed=42)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
valid_ds = (
    to_tf_dataset("validation")
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
from transformers import create_optimizer

# === F. Muat, Compile, dan Fine‑Tune Model ===
model = TFAutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3  # jumlah kelas sesuai mapping ClassLabel
)

# Hitung jumlah langkah training
steps_per_epoch = len(train_ds)  # biasanya = jumlah batch
num_train_steps = steps_per_epoch * 3  # total epoch

# Buat optimizer & scheduler
optimizer, schedule = create_optimizer(
    init_lr=5e-5,                # learning rate awal
    num_train_steps=num_train_steps,
    num_warmup_steps=0           # bisa dicoba juga 10% dari num_train_steps
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])
model.fit(train_ds, validation_data=valid_ds, epochs=3)

tf_model.h5:   0%|          | 0.00/656M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3
925/925 [==============================] - 455s 441ms/step - loss: 0.2895 - accuracy: 0.9277 - val_loss: 0.2655 - val_accuracy: 0.9404
Epoch 2/3
925/925 [==============================] - 405s 438ms/step - loss: 0.2658 - accuracy: 0.9368 - val_loss: 0.2586 - val_accuracy: 0.9404
Epoch 3/3
925/925 [==============================] - 405s 438ms/step - loss: 0.2975 - accuracy: 0.9288 - val_loss: 0.2551 - val_accuracy: 0.9404


In [ ]:
!pip install tensorflowjs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
db-dtypes 1.4.3 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
google-cloud-bigquery 3.32.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
INFO: pip is looking at multiple versions of onnx to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 110.8 MB/s eta 0:00:00
   

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.4 MB/s eta 0:00:00


In [ ]:
# === G. Simpan Model ke Berbagai Format ===
# SavedModel
saved_model_dir = "./indobert_tf_savedmodel"
model.save(saved_model_dir, save_format="tf")

# HDF5
h5_path = "./indobert_tf_model.h5"
model.save_weights(h5_path, save_format="h5")

# TF Lite
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
tflite_model = converter.convert()
with open("./indobert_model.tflite", "wb") as f:
    f.write(tflite_model)

# TFJS
!python -m tensorflowjs.converters.converter \
  --input_format=keras \
  ./indobert_tf_model.h5 \
  ./indobert_tfjs_model

# **Inference**

In [ ]:
import numpy as np
import onnxruntime as ort
import time
from scipy.special import softmax
from tqdm import tqdm

# === Load CSV ===
xlsx_infer_path = "/content/inference.xlsx"
df_test = pd.read_excel(xlsx_infer_path)
texts = df_test["Description"].tolist()

In [ ]:
# === Inisialisasi Tokenizer dan Label Map ===
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
label_map = {0: "HOAX", 1: "BELUM DIVERIFIKASI", 2: "FAKTA"}

In [ ]:
# === 1. TensorFlow Inference ===
print("\n🔍 Inference with TensorFlow Model")
saved_model_dir = "./indobert_tf_savedmodel"
model = tf.keras.models.load_model(saved_model_dir)

# Tokenisasi
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="tf"
)

# Hitung waktu
start_tf = time.time()

# Inference dengan progress bar
logits_tf = []
batch_size = 16
for i in tqdm(range(0, len(texts), batch_size), desc="TF Inference"):
    input_batch = {k: v[i:i+batch_size] for k, v in inputs.items()}
    output = model(input_batch)
    logits_tf.append(output["logits"].numpy())  # Perhatikan bagian ini!

# Gabungkan hasil logit dan hitung probabilitas & prediksi
logits_tf = np.concatenate(logits_tf, axis=0)
probs_tf = softmax(logits_tf, axis=1)
preds_tf = np.argmax(probs_tf, axis=1)

df_test["prediction_tf"] = [label_map[int(p)] for p in preds_tf]
df_test["prob_tf_HOAX"] = probs_tf[:, 0]
df_test["prob_tf_BELUM"] = probs_tf[:, 1]
df_test["prob_tf_FAKTA"] = probs_tf[:, 2]

end_tf = time.time()
print(f"✅ TensorFlow inference selesai dalam {end_tf - start_tf:.2f} detik.")


🔍 Inference with TensorFlow Model


TF Inference: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]

✅ TensorFlow inference selesai dalam 1.04 detik.


In [ ]:
df_test.to_excel("result.xlsx", index=False)